In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib_inline.backend_inline as inline
inline.set_matplotlib_formats('svg')

In [14]:
import seaborn as sns
iris = sns.load_dataset('iris')
# Organize data
# Convert data from pandas dataframe to pytorch tensors
data = torch.tensor( iris[iris.columns[0:4]].values ).float()

# transform species to number/label encoding
labels = torch.zeros(len(iris), dtype=torch.long)
# labels[iris['species'] == 'setosa'] = 0    This is commented out because the default value of labels is already 0, so we don't need to assign it again.
labels[iris['species'] == 'versicolor'] = 1
labels[iris['species'] == 'virginica'] = 2


In [18]:
# Create model class

class ANNiris(nn.Module):
    def __init__(self,n_units, n_layers):
        super().__init__() 
        # Create dictionary to store layers
        self.layers = nn.ModuleDict()
        self.n_layers = n_layers

        # input layer
        self.layers['input'] = nn.Linear(4, n_units)

        # hidden layers
        for i in range(n_layers):
            self.layers[f'hidden_{i}'] = nn.Linear(n_units, n_units)

        # output layer
        self.layers['output'] = nn.Linear(n_units, 3)

    
    # forward pass
    def forward(self, x):
        # input layer
        x = self.layers['input'](x)

        # hidden layers
        for i in range(self.n_layers):
            x = F.relu(self.layers[f'hidden_{i}'](x))

        # Return output layer
        x = self.layers['output'](x)
        return x


In [19]:
net = ANNiris(n_units = 12, n_layers = 4)
net

ANNiris(
  (layers): ModuleDict(
    (input): Linear(in_features=4, out_features=12, bias=True)
    (hidden_0): Linear(in_features=12, out_features=12, bias=True)
    (hidden_1): Linear(in_features=12, out_features=12, bias=True)
    (hidden_2): Linear(in_features=12, out_features=12, bias=True)
    (hidden_3): Linear(in_features=12, out_features=12, bias=True)
    (output): Linear(in_features=12, out_features=3, bias=True)
  )
)

In [ ]:
# Test the model with a forward pass

temp = torch.randn(10,4)

y = net(temp)

print(y.shape), print(' ')

print('Output of the model: \n', y)

torch.Size([100, 3])
 
Output of the model: 
 tensor([[-0.1740,  0.0860, -0.3453],
        [-0.1717,  0.0831, -0.3454],
        [-0.1505,  0.0836, -0.3513],
        [-0.1704,  0.0869, -0.3418],
        [-0.1732,  0.0861, -0.3462],
        [-0.1714,  0.0869, -0.3418],
        [-0.1697,  0.0859, -0.3450],
        [-0.1473,  0.0845, -0.3553],
        [-0.1533,  0.0870, -0.3558],
        [-0.1717,  0.0867, -0.3414],
        [-0.1720,  0.0812, -0.3407],
        [-0.1679,  0.0873, -0.3477],
        [-0.1647,  0.0875, -0.3563],
        [-0.1551,  0.0874, -0.3593],
        [-0.1657,  0.0872, -0.3459],
        [-0.1737,  0.0861, -0.3430],
        [-0.1700,  0.0784, -0.3439],
        [-0.1699,  0.0876, -0.3446],
        [-0.1583,  0.0864, -0.3495],
        [-0.1723,  0.0842, -0.3421],
        [-0.1649,  0.0866, -0.3465],
        [-0.1610,  0.0874, -0.3580],
        [-0.1740,  0.0853, -0.3429],
        [-0.1721,  0.0804, -0.3401],
        [-0.1713,  0.0874, -0.3433],
        [-0.1614,  0.0864, -0

In [ ]:
# Train model
def train_model(model, epochs):

    loss_fun = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

    for epoch in range(epochs):
        # Forward pass
        y_hat = model(data)

        # Compute loss
        loss = loss_fun(y_hat, labels)

        # Backward propagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # final foward pass
    predictions = model(data)
    predicted_labels = torch.argmax(predictions, dim=1) # Get the index of the maximum value in each row (predicted class)
    accuracy = 100*torch.mean((predicted_labels == labels).float()) # Calculate accuracy

    # Total number of trainable parameters
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    return loss, accuracy, total_params